In [2]:
import pandas as pd
import numpy as np
from collections import defaultdict
from reactiva.config import DATASET_URI
from datetime import datetime
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


### Importando el dataset

In [3]:

df = pd.read_csv(DATASET_URI)

In [5]:
df.groupby('Customer ID')

### procesamiento de datos pequeño para agregar session 

In [24]:
df['Purchase Date'] = pd.to_datetime(df['Purchase Date'])
df['season'] = df['Purchase Date'].dt.month.apply(lambda x: 'winter' if x in(12,1,2) else 'summer' if x in (3,4,5) else 'monsoon' if x in (6,7,8,9) else 'post-monsoon')
df_tovectorize = df[['Age','Gender','Location','season','Brand','Category','Online/Offline','Customer ID','Item Purchased','Purchase Date']]


df_purchases_270morethandays = df[df['Purchase Date']<=(df['Purchase Date'].max() - pd.Timedelta(days=270))]
df_purchases_270lessthandays = df[df['Purchase Date']>(df['Purchase Date'].max() - pd.Timedelta(days=270))]
cx_didnot_270daysago =np.setdiff1d(df_purchases_270morethandays['Customer ID'].unique(), df_purchases_270lessthandays['Customer ID'].unique())

# trayendo la temporada actual# 
month = pd.Timestamp.now().month

season = (
    'winter' if month in (12, 1, 2)
    else 'summer' if month in (3, 4, 5)
    else 'monsoon' if month in (6, 7, 8, 9)
    else 'post-monsoon')

season_list = pd.Series(['winter','summer','monsoon','post-monsoon'])
list_season = season_list[season_list != season]

### modelo de recomendación basado en usuari, pronóstico de compra basado en similitud de coseno de temporada anterior extrapolado and temporada actual, features de vector:
    Items Purchased
se crean los buckets basado en los usuarios port estaciones y sobre eso comparamos la similitud

In [52]:
def precision_recall_at_k(recommended, actual, k):
    rec_k = recommended[:k]
    if not rec_k:
        return 0.0, 0.0, 0.0
    hits = len(set(rec_k) & actual)
    precision = hits / len(rec_k)
    recall = (hits / len(actual) if actual else 0.0)
    hit_rate = float(hits > 0)
    return precision, recall, hit_rate

def backtest_recommender_pr(df, holdout_season, season_list, k=5):
    train_seasons = season_list[season_list != holdout_season]
    df_train = df[df['season'].isin(train_seasons)]
    df_holdout_actual = df[df['season'] == holdout_season]

    train_customers = df_train['Customer ID'].unique()
    holdout_customers = df_holdout_actual['Customer ID'].unique()
    scoring_customers = np.intersect1d(train_customers, holdout_customers)

    session_matrices = {}
    sparsities = []

    for s in train_seasons:
        df_tovector = df_train[df_train['season'] == s]
        user_item_matrix = pd.crosstab(df_tovector['Customer ID'], df_tovector['Item Purchased'])

        sparsity = 1 - ((user_item_matrix.values > 0).sum() / user_item_matrix.size)
        sparsities.append(sparsity)

        similarity = cosine_similarity(user_item_matrix)
        session_matrices[s] = pd.DataFrame(
            similarity, index=user_item_matrix.index, columns=user_item_matrix.index
        )

    print(f'Sparsity: {np.mean(sparsities):.3f}')

    results = []
    for user in scoring_customers:
        user_dict = {}
        for s, similarity_df in session_matrices.items():
            if user not in similarity_df.columns:
                continue
            top5 = similarity_df[user].drop(user).sort_values(ascending=False).head(5)
            for sim_user, l in top5.items():
                if sim_user not in df_holdout_actual['Customer ID'].values:
                    continue
                user_dict[sim_user] = df_holdout_actual[df_holdout_actual['Customer ID'] == sim_user]

        if len(user_dict) > 1:
            df_users = pd.concat(user_dict.values(), ignore_index=True)
            recommendation = df_users['Item Purchased'].value_counts().head(k).index.tolist()
        else:
            recommendation = []

        actual = set(df_holdout_actual[df_holdout_actual['Customer ID'] == user]['Item Purchased'])
        precision, recall, hit_rate = precision_recall_at_k(recommendation, actual, k)

        results.append({
            'Customer ID': user,
            'recommended': recommendation,
            'actual': list(actual),
            'precision@k': precision,
            'recall@k': recall,
            'hitting_rate': hit_rate
        })

    results_df = pd.DataFrame(results)
    print(f'Precision@{k}: {results_df["precision@k"].mean():.3f}')
    print(f'Recall@{k}:    {results_df["recall@k"].mean():.3f}')
    print(f'hitting rate :{results_df["hitting_rate"].mean():.3f}')
    return results_df

In [54]:
for i in season_list:

    results_df = backtest_recommender_pr(
        df,
        holdout_season=i,
        season_list=season_list,
        k=5
    )

Sparsity: 0.942
Precision@5: 0.072
Recall@5:    0.226
hitting rate :0.285
Sparsity: 0.943
Precision@5: 0.075
Recall@5:    0.227
hitting rate :0.295
Sparsity: 0.944
Precision@5: 0.076
Recall@5:    0.233
hitting rate :0.315
Sparsity: 0.941
Precision@5: 0.055
Recall@5:    0.176
hitting rate :0.210


### Modelo baso en usuario con balance de frequencias
    precision:
    Recall:
    sparsity:

    se busca determinar si la frecuencia impacta el performance


In [72]:


# ============================================================
# 1. BUILD CUSTOMER PROFILE
#    Frequency matters: 4 Jackets > 1 Jacket
# ============================================================

def build_customer_profile(df_train):

    customer_item_matrix = (
        df_train
        .groupby(['Customer ID', 'Item Purchased'])
        .size()
        .unstack(fill_value=0)
    )

    return customer_item_matrix


# ============================================================
# 2. BUILD CUSTOMER SIMILARITY
# ============================================================

def build_customer_similarity(df_train):

    customer_item_matrix = build_customer_profile(
        df_train
    )

    similarity = cosine_similarity(
        customer_item_matrix
    )

    similarity_df = pd.DataFrame(
        similarity,
        index=customer_item_matrix.index,
        columns=customer_item_matrix.index
    )

    return similarity_df


# ============================================================
# 3. RECOMMEND FROM SIMILAR CUSTOMERS
# ============================================================

def get_user_based_recommendations(
    customer_id,
    similarity_df,
    df_holdout,
    top_n=5,
    k=5
):

    # Customer must exist in the training profile
    if customer_id not in similarity_df.index:
        return []

    # --------------------------------------------------------
    # Find most similar customers
    # --------------------------------------------------------

    neighbors = (
        similarity_df[customer_id]
        .drop(customer_id)
        .sort_values(ascending=False)
        .head(top_n)
    )

    if neighbors.empty:
        return []

    # --------------------------------------------------------
    # Get future purchases made by those neighbors
    # --------------------------------------------------------

    neighbor_ids = neighbors.index

    neighbor_purchases = df_holdout[
        df_holdout['Customer ID'].isin(neighbor_ids)
    ]

    if neighbor_purchases.empty:
        return []

    # --------------------------------------------------------
    # Rank items by frequency among similar customers
    # --------------------------------------------------------

    item_counts = (
        neighbor_purchases['Item Purchased']
        .value_counts()
    )

    # --------------------------------------------------------
    # Do not recommend items the target customer already
    # purchased during training
    # --------------------------------------------------------

    return item_counts.head(k).index.tolist()


# ============================================================
# 4. PRECISION / RECALL / HIT RATE
# ============================================================

def precision_recall_at_k(
    recommended,
    actual,
    k
):

    rec_k = recommended[:k]

    if not rec_k:
        return 0.0, 0.0, 0.0

    hits = len(
        set(rec_k) & set(actual)
    )

    precision = (
        hits / len(rec_k)
    )

    recall = (
        hits / len(actual)
        if actual
        else 0.0
    )

    hit_rate = float(
        hits > 0
    )

    return precision, recall, hit_rate


# ============================================================
# 5. SPARSITY
# ============================================================

def calculate_sparsity(customer_item_matrix):

    if customer_item_matrix.size == 0:
        return 0.0

    non_zero = (
        customer_item_matrix.values > 0
    ).sum()

    return 1 - (
        non_zero /
        customer_item_matrix.size
    )


# ============================================================
# 6. SEASONAL BACKTEST
# ============================================================

def backtest_frequency_user_based(
    df,
    holdout_season,
    season_list,
    top_n=5,
    k=5
):

    # --------------------------------------------------------
    # TRAIN = all seasons except holdout
    # --------------------------------------------------------

    train_seasons = season_list[
        season_list != holdout_season
    ]

    df_train = df[
        df['season'].isin(train_seasons)
    ]

    # --------------------------------------------------------
    # HOLDOUT = current/future season
    # --------------------------------------------------------

    df_holdout = df[
        df['season'] == holdout_season
    ]

    # --------------------------------------------------------
    # Only customers existing in both periods
    # --------------------------------------------------------

    train_customers = (
        df_train['Customer ID'].unique()
    )

    holdout_customers = (
        df_holdout['Customer ID'].unique()
    )

    scoring_customers = np.intersect1d(
        train_customers,
        holdout_customers
    )

    # --------------------------------------------------------
    # Build relationship ONCE
    # --------------------------------------------------------

    customer_item_matrix = (
        build_customer_profile(
            df_train
        )
    )

    similarity_df = (
        build_customer_similarity(
            df_train
        )
    )

    # --------------------------------------------------------
    # Sparsity
    # --------------------------------------------------------

    sparsity = calculate_sparsity(
        customer_item_matrix
    )

    print(
        f"Sparsity: {sparsity:.4f}"
    )

    # --------------------------------------------------------
    # Evaluate customers
    # --------------------------------------------------------

    results = []

    precision_scores = []
    recall_scores = []
    hit_scores = []

    for customer_id in scoring_customers:

        # ----------------------------------------------------
        # Recommendations based ONLY on relationships learned
        # from the training seasons
        # ----------------------------------------------------

        recommendation = (
            get_user_based_recommendations(
                customer_id=customer_id,
                similarity_df=similarity_df,
                df_holdout=df_holdout,
                top_n=top_n,
                k=k
            )
        )

        # ----------------------------------------------------
        # Actual purchases in holdout season
        # ----------------------------------------------------

        actual = set(
            df_holdout[
                df_holdout['Customer ID'] == customer_id
            ]['Item Purchased']
        )

        precision, recall, hit_rate = (
            precision_recall_at_k(
                recommendation,
                actual,
                k
            )
        )

        precision_scores.append(
            precision
        )

        recall_scores.append(
            recall
        )

        hit_scores.append(
            hit_rate
        )

        results.append({
            'Customer ID': customer_id,
            'recommended': recommendation,
            'actual': list(actual),
            'precision@k': precision,
            'recall@k': recall,
            'hitting_rate': hit_rate
        })

    # --------------------------------------------------------
    # Average metrics
    # --------------------------------------------------------

    results_df = pd.DataFrame(
        results
    )

    print(
        f'Precision@{k}: '
        f'{np.mean(precision_scores):.3f}'
    )

    print(
        f'Recall@{k}:    '
        f'{np.mean(recall_scores):.3f}'
    )

    print(
        f'HitRate@{k}:   '
        f'{np.mean(hit_scores):.3f}'
    )

    return (
        results_df,
        similarity_df,
        customer_item_matrix
    )

In [75]:
# tryting the model #
results, similarity, customer_profile = (
    backtest_frequency_user_based(
        df=df,
        holdout_season='winter',
        season_list=season_list,
        top_n=5,
        k=5
    )
)

Sparsity: 0.9033
Precision@5: 0.062
Recall@5:    0.149
HitRate@5:   0.192


### Modelo Conetent-based

In [76]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd


def build_content_profiles(df_train):
    """
    Build a text profile for each customer from their historical
    item purchases. Repeated purchases are preserved.
    """

    customer_profiles = (
        df_train
        .groupby('Customer ID')['Item Purchased']
        .apply(lambda x: ' '.join(x.astype(str)))
    )

    vectorizer = TfidfVectorizer()

    customer_vectors = vectorizer.fit_transform(
        customer_profiles
    )

    similarity = cosine_similarity(
        customer_vectors
    )

    similarity_df = pd.DataFrame(
        similarity,
        index=customer_profiles.index,
        columns=customer_profiles.index
    )

    return similarity_df, customer_vectors, vectorizer


def precision_recall_at_k(recommended, actual, k):

    rec_k = recommended[:k]

    if not rec_k:
        return 0.0, 0.0, 0.0

    hits = len(
        set(rec_k) & set(actual)
    )

    precision = hits / len(rec_k)

    recall = (
        hits / len(actual)
        if actual
        else 0.0
    )

    hit_rate = float(hits > 0)

    return precision, recall, hit_rate


def backtest_content_based(
    df,
    holdout_season,
    season_list,
    top_n=5,
    k=5
):

    # --------------------------------------------------------
    # TRAIN = past seasons
    # --------------------------------------------------------

    train_seasons = season_list[
        season_list != holdout_season
    ]

    df_train = df[
        df['season'].isin(train_seasons)
    ]

    # --------------------------------------------------------
    # HOLDOUT = current season
    # --------------------------------------------------------

    df_holdout_actual = df[
        df['season'] == holdout_season
    ]

    train_customers = (
        df_train['Customer ID'].unique()
    )

    holdout_customers = (
        df_holdout_actual['Customer ID'].unique()
    )

    scoring_customers = np.intersect1d(
        train_customers,
        holdout_customers
    )

    # --------------------------------------------------------
    # BUILD CUSTOMER CONTENT PROFILES ON PAST DATA ONLY
    # --------------------------------------------------------

    similarity_df, customer_vectors, vectorizer = (
        build_content_profiles(df_train)
    )

    # --------------------------------------------------------
    # SPARSITY
    # --------------------------------------------------------

    customer_item_matrix = pd.crosstab(
        df_train['Customer ID'],
        df_train['Item Purchased']
    )

    sparsity = 1 - (
        (customer_item_matrix.values > 0).sum()
        / customer_item_matrix.size
    )

    print(f'Sparsity: {sparsity:.3f}')

    results = []

    precision_scores = []
    recall_scores = []
    hit_scores = []

    # --------------------------------------------------------
    # EVALUATE EACH CUSTOMER
    # --------------------------------------------------------

    for customer_id in scoring_customers:

        if customer_id not in similarity_df.index:
            continue

        # ----------------------------------------------------
        # Find customers with similar purchase profiles
        # ----------------------------------------------------

        similar_customers = (
            similarity_df[customer_id]
            .drop(customer_id)
            .sort_values(
                ascending=False
            )
            .head(top_n)
        )

        if similar_customers.empty:
            continue

        # ----------------------------------------------------
        # Get what those similar customers bought
        # in the holdout season
        # ----------------------------------------------------

        neighbor_ids = similar_customers.index

        neighbor_purchases = df_holdout_actual[
            df_holdout_actual['Customer ID'].isin(
                neighbor_ids
            )
        ]

        if neighbor_purchases.empty:
            recommendation = []

        else:

            # Rank by frequency of purchase among
            # the similar customers
            recommendation = (
                neighbor_purchases[
                    'Item Purchased'
                ]
                .value_counts()
                .head(k)
                .index
                .tolist()
            )

        # ----------------------------------------------------
        # Actual purchases of target customer
        # ----------------------------------------------------

        actual = set(
            df_holdout_actual[
                df_holdout_actual['Customer ID'] == customer_id
            ]['Item Purchased']
        )

        precision, recall, hit_rate = (
            precision_recall_at_k(
                recommendation,
                actual,
                k
            )
        )

        precision_scores.append(
            precision
        )

        recall_scores.append(
            recall
        )

        hit_scores.append(
            hit_rate
        )

        results.append({
            'Customer ID': customer_id,
            'Similar Customers': similar_customers.index.tolist(),
            'Recommendations': recommendation,
            'Actual': list(actual),
            f'Precision@{k}': precision,
            f'Recall@{k}': recall,
            f'HitRate@{k}': hit_rate
        })

    results_df = pd.DataFrame(results)

    print(
        f'Precision@{k}: '
        f'{np.mean(precision_scores):.3f}'
    )

    print(
        f'Recall@{k}:    '
        f'{np.mean(recall_scores):.3f}'
    )

    print(
        f'HitRate@{k}:   '
        f'{np.mean(hit_scores):.3f}'
    )

    return results_df, similarity_df, vectorizer

In [77]:
season_list = np.array([
    'winter',
    'summer',
    'monsoon',
    'post-monsoon'
])

results, similarity_df, vectorizer = backtest_content_based(
    df,
    holdout_season='winter',
    season_list=season_list,
    top_n=5,
    k=5
)

Sparsity: 0.903
Precision@5: 0.067
Recall@5:    0.150
HitRate@5:   0.194


### Creando un modelo de recomendación CF hybrid user y popularity
    Precisión@:
    Recall@:
    Hitting_rate:

In [66]:

def precision_recall_at_k(recommended, actual, k):
    rec_k = recommended[:k]
    if not rec_k:
        return 0.0, 0.0, 0.0
    hits = len(set(rec_k) & actual)
    precision = hits / len(rec_k)
    recall = (hits / len(actual) if actual else 0.0)
    hit_rate = float(hits > 0)
    return precision, recall, hit_rate


def get_popularity_scores(df_train):
    """Item popularity across the full train set, normalized 0-1."""
    counts = df_train['Item Purchased'].value_counts()
    return counts / counts.max()


def get_cf_scores(user, season_matrices, df_holdout_actual):
    """Neighbor-frequency scores for a single user, normalized 0-1."""
    user_dict = {}
    for s, similarity_df in season_matrices.items():
        if user not in similarity_df.columns:
            continue
        top5 = similarity_df[user].drop(user).sort_values(ascending=False).head(5)
        for sim_user, l in top5.items():
            if sim_user not in df_holdout_actual['Customer ID'].values:
                continue
            user_dict[sim_user] = df_holdout_actual[df_holdout_actual['Customer ID'] == sim_user]

    if len(user_dict) < 1:
        return pd.Series(dtype=float)

    df_users = pd.concat(user_dict.values(), ignore_index=True)
    counts = df_users['Item Purchased'].value_counts()
    return counts / counts.max() if len(counts) else pd.Series(dtype=float)


def sparsity_of_matrix(user_item_matrix):
    """Fraction of zero cells in a user-item matrix. 1.0 = fully empty."""
    total_cells = user_item_matrix.size
    if total_cells == 0:
        return np.nan
    nonzero_cells = (user_item_matrix.values > 0).sum()
    return 1 - (nonzero_cells / total_cells)


def backtest_hybrid_avg_threshold(df, holdout_season, season_list, k=5, top_n=5):
    """
    Threshold-gated hybrid, fixed at 100% average similarity:

    - For each customer, find their top-N most similar neighbors in each
      training-season similarity matrix, pool those similarity scores,
      and average them.
    - Only if that average similarity is a PERFECT 1.0 (100%) does the
      customer get a CF-based recommendation (neighbor-frequency scores).
      Any customer below 100% falls back to popularity of the training
      seasons.
    - Also reports Precision@k, Recall@k, and user-item matrix sparsity
      per training season.
    """
    THRESHOLD = 1.0

    train_seasons = season_list[season_list != holdout_season]
    df_train = df[df['season'].isin(train_seasons)]
    df_holdout_actual = df[df['season'] == holdout_season]

    train_customers = df_train['Customer ID'].unique()
    holdout_customers = df_holdout_actual['Customer ID'].unique()
    scoring_customers = np.intersect1d(train_customers, holdout_customers)

    season_matrices = {}
    sparsity_by_season = {}
    for s in train_seasons:
        df_tovector = df_train[df_train['season'] == s]
        user_item_matrix = pd.crosstab(df_tovector['Customer ID'], df_tovector['Item Purchased'])
        sparsity_by_season[s] = sparsity_of_matrix(user_item_matrix)
        similarity = cosine_similarity(user_item_matrix)
        season_matrices[s] = pd.DataFrame(
            similarity, index=user_item_matrix.index, columns=user_item_matrix.index
        )

    pop_scores = get_popularity_scores(df_train)

    results = []
    cf_used, pop_used = 0, 0
    for user in scoring_customers:
        # average similarity across this customer's top-N neighbors,
        # pooled across every training-season similarity matrix
        all_top_neighbor_sims = []
        for s, similarity_df in season_matrices.items():
            if user not in similarity_df.columns:
                continue
            neighbor_sims = similarity_df[user].drop(user).sort_values(ascending=False).head(top_n)
            all_top_neighbor_sims.extend(neighbor_sims.tolist())
        avg_sim = np.mean(all_top_neighbor_sims) if all_top_neighbor_sims else -1

        if avg_sim >= THRESHOLD:
            cf_scores = get_cf_scores(user, season_matrices, df_holdout_actual)
            if len(cf_scores):
                recommendation = cf_scores.sort_values(ascending=False).head(k).index.tolist()
                cf_used += 1
            else:
                recommendation = pop_scores.head(k).index.tolist()
                pop_used += 1
        else:
            recommendation = pop_scores.head(k).index.tolist()
            pop_used += 1

        actual = set(df_holdout_actual[df_holdout_actual['Customer ID'] == user]['Item Purchased'])
        precision, recall, hit_rate = precision_recall_at_k(recommendation, actual, k)
        results.append({
            'Customer ID': user,
            'avg_neighbor_sim': avg_sim,
            'precision@k': precision,
            'recall@k': recall,
            'hitting_rate': hit_rate
        })

    results_df = pd.DataFrame(results)
    overall_sparsity = np.mean(list(sparsity_by_season.values()))

    print(f'threshold = {THRESHOLD} (avg of top-{top_n} neighbors, exact matches only)')
    print(f'Precision@{k}: {results_df["precision@k"].mean():.4f}')
    print(f'Recall@{k}:    {results_df["recall@k"].mean():.4f}')
    print(f'hitting rate : {results_df["hitting_rate"].mean():.4f}')
    print(f'customers routed to CF:                {cf_used}')
    print(f'customers routed to popularity fallback: {pop_used}')
    print()
    print('sparsity by season:')
    for s, sp in sparsity_by_season.items():
        print(f'  {s}: {sp:.4f}')
    print(f'overall sparsity (avg across seasons): {overall_sparsity:.4f}')

    return results_df


# usage:
# df['season'] = df['Purchase Date'].dt.month.apply(
#     lambda x: 'winter' if x in (12,1,2) else 'summer' if x in (3,4,5)
#     else 'monsoon' if x in (6,7,8,9) else 'post-monsoon'
# )
# season_list = pd.Series(['winter','summer','monsoon','post-monsoon'])
# r = backtest_hybrid_avg_threshold(df, 'winter', season_list=season_list, k=5, top_n=5)

In [67]:
##trying the model above
backtest_hybrid_avg_threshold(df, 'winter', season_list=season_list, k=5, top_n=5)


threshold = 1.0 (avg of top-5 neighbors, exact matches only)
Precision@5: 0.0909
Recall@5:    0.3041
hitting rate : 0.3695
customers routed to CF:                830
customers routed to popularity fallback: 702

sparsity by season:
  summer: 0.9422
  monsoon: 0.9367
  post-monsoon: 0.9484
overall sparsity (avg across seasons): 0.9424


,Customer ID,avg_neighbor_sim,precision@k,recall@k,hitting_rate
0,CUST000006,1.000000,0.0,0.0,0.0
1,CUST000008,0.921895,0.0,0.0,0.0
2,CUST000011,1.000000,0.0,0.0,0.0
3,CUST000014,0.853553,0.4,1.0,1.0
4,CUST000015,1.000000,0.0,0.0,0.0
...,...,...,...,...,...
1527,CUST003492,1.000000,0.0,0.0,0.0
1528,CUST003493,0.878282,0.2,0.5,1.0
1529,CUST003495,1.000000,0.0,0.0,0.0
1530,CUST003497,1.000000,0.0,0.0,0.0


### modelo de recomendación basado en poularidad, metricas:
    Precision:
    Recall:
    Sparsity:

In [64]:
def get_popularity_scores(df_train):
    """Item popularity across the full train set, normalized 0-1."""
    counts = df_train['Item Purchased'].value_counts()
    return counts / counts.max()


def pprecision_recall_at_k(recommended, actual, k):
    """
    recommended: list of recommended items (already ranked, best first)
    actual: set of items the customer actually bought
    k: cutoff
    """
    rec_k = recommended[:k]

    if not rec_k:
        return 0.0, 0.0

    hits = len(set(rec_k) & actual)

    precision = hits / len(rec_k)

    recall = (
        hits / len(actual)
        if actual
        else 0.0
    )

    return precision, recall


def backtest_popularity(df, holdout_season, season_list, k=5):

    train_seasons = season_list[season_list != holdout_season]
    df_train = df[df['season'].isin(train_seasons)]
    df_holdout_actual = df[df['season'] == holdout_season]

    train_customers = df_train['Customer ID'].unique()
    holdout_customers = df_holdout_actual['Customer ID'].unique()
    scoring_customers = np.intersect1d(
        train_customers,
        holdout_customers
    )

    # --------------------------------------------------------
    # Sparsity from Customer × Item matrix
    # --------------------------------------------------------

    user_item_matrix = pd.crosstab(
        df_train['Customer ID'],
        df_train['Item Purchased']
    )

    sparsity = 1 - (
        (user_item_matrix.values > 0).sum()
        / user_item_matrix.size
    )

    print(f'Sparsity: {sparsity:.3f}')

    # --------------------------------------------------------
    # Popularity scores
    # --------------------------------------------------------

    pop_scores = get_popularity_scores(df_train)

    results = []

    for user in scoring_customers:

        recommendation = (
            pop_scores
            .sort_values(ascending=False)
            .head(k)
            .index
            .tolist()
        )

        actual = set(
            df_holdout_actual[
                df_holdout_actual['Customer ID'] == user
            ]['Item Purchased']
        )

        precision, recall = pprecision_recall_at_k(
            recommendation,
            actual,
            k
        )

        results.append({
            'Customer ID': user,
            'recommended': recommendation,
            'actual': list(actual),
            'precision@k': precision,
            'recall@k': recall
        })

    results_df = pd.DataFrame(results)

    print(
        f'Precision@{k}: '
        f'{results_df["precision@k"].mean():.3f}'
    )

    print(
        f'Recall@{k}:    '
        f'{results_df["recall@k"].mean():.3f}'
    )

    return results_df

In [62]:
#testing model popularity#
results = backtest_popularity(
    df,
    holdout_season='winter',
    season_list=season_list,
    k=5
)

Sparsity: 0.903
Precision@5: 0.108
Recall@5:    0.393


### Creando un modelo de recomendación CF basado en items
    Precisión:
    Recall:
    Sparsity:

El modelo crea la matrix  de relación de items y despúes recomienda los items que aparecen con mayor frecuencia cuando un item es comprado

In [30]:
# ============================================================
# 1. TEMPORAL TRAIN / TEST SPLIT
# ============================================================

def temporal_train_test_split(
    df,
    date_col="Purchase Date",
    test_size=0.20
):

    df = df.copy()

    df[date_col] = pd.to_datetime(df[date_col])

    cutoff = df[date_col].quantile(1 - test_size)

    df_train = df[df[date_col] <= cutoff].copy()
    df_test = df[df[date_col] > cutoff].copy()

    return df_train, df_test


# ============================================================
# 2. BUILD ITEM-ITEM SIMILARITY
# ============================================================

def build_item_item_similarity(df_train):

    # --------------------------------------------------------
    # Each customer contributes UNIQUE items
    # --------------------------------------------------------

    customer_items = (
        df_train
        .groupby("Customer ID")["Item Purchased"]
        .apply(set)
    )

    # --------------------------------------------------------
    # Create all items
    # --------------------------------------------------------

    all_items = sorted(
        df_train["Item Purchased"].dropna().unique()
    )

    # --------------------------------------------------------
    # Customer x Item binary matrix
    #
    # 1 = customer purchased item
    # 0 = customer did not purchase item
    # --------------------------------------------------------

    customer_item_matrix = pd.DataFrame(
        0,
        index=customer_items.index,
        columns=all_items,
        dtype=int
    )

    for customer_id, items in customer_items.items():

        for item in items:

            customer_item_matrix.loc[
                customer_id,
                item
            ] = 1

    # --------------------------------------------------------
    # SPARSITY
    # --------------------------------------------------------

    total_cells = customer_item_matrix.size

    non_zero_cells = customer_item_matrix.to_numpy().sum()

    sparsity = (
        1 - (non_zero_cells / total_cells)
        if total_cells > 0
        else 0.0
    )

    # --------------------------------------------------------
    # ITEM x CUSTOMER matrix
    #
    # Each item is represented by the customers who bought it
    # --------------------------------------------------------

    item_customer_matrix = customer_item_matrix.T

    # --------------------------------------------------------
    # ONE similarity calculation
    # --------------------------------------------------------

    similarity = cosine_similarity(
        item_customer_matrix
    )

    similarity = pd.DataFrame(
        similarity,
        index=item_customer_matrix.index,
        columns=item_customer_matrix.index
    )

    return similarity, sparsity


# ============================================================
# 3. GET RECOMMENDATIONS
# ============================================================

def get_recommendations(
    trigger_item,
    similarity,
    top_n=5
):

    if trigger_item not in similarity.index:
        return []

    scores = similarity.loc[trigger_item].copy()

    # Do not recommend the trigger itself
    scores = scores.drop(
        labels=[trigger_item],
        errors="ignore"
    )

    # Remove zero-similarity items
    scores = scores[scores > 0]

    # Highest similarity first
    scores = scores.sort_values(
        ascending=False
    )

    return scores.head(top_n).index.tolist()


# ============================================================
# 4. PRECISION@K
# ============================================================

def precision_at_k(
    recommendations,
    actual_items,
    k
):

    recommendations = recommendations[:k]

    if not recommendations:
        return 0.0

    hits = len(
        set(recommendations) &
        set(actual_items)
    )

    return hits / len(recommendations)


# ============================================================
# 5. TRUE RECALL@K
# ============================================================

def recall_at_k(
    recommendations,
    actual_items,
    k
):

    recommendations = recommendations[:k]

    actual_items = set(actual_items)

    if not actual_items:
        return 0.0

    hits = len(
        set(recommendations) &
        actual_items
    )

    return hits / len(actual_items)


# ============================================================
# 6. HIT RATE@K
#
# Used when we have ONE specific future item.
# ============================================================

def hit_rate_at_k(
    recommendations,
    actual_item,
    k
):

    return int(
        actual_item in recommendations[:k]
    )


# ============================================================
# 7. EVALUATION
# ============================================================

def evaluate_item_item_cf(
    df_data,
    k=5,
    test_size=0.20
):

    results = []

    precisions = []
    recalls = []
    hit_rates = []

    # --------------------------------------------------------
    # Sort chronologically
    # --------------------------------------------------------

    df_data = df_data.copy()

    df_data["Purchase Date"] = pd.to_datetime(
        df_data["Purchase Date"]
    )

    df_data = df_data.sort_values(
        "Purchase Date"
    )

    # --------------------------------------------------------
    # Temporal split
    # --------------------------------------------------------

    df_train, df_test = temporal_train_test_split(
        df_data,
        test_size=test_size
    )

    # --------------------------------------------------------
    # Build ONE global similarity model
    # --------------------------------------------------------

    similarity, sparsity = build_item_item_similarity(
        df_train
    )

    print(f"Sparsity: {sparsity:.4f}")
    print(f"Sparsity: {sparsity * 100:.2f}%")

    # --------------------------------------------------------
    # Evaluate each customer
    # --------------------------------------------------------

    for customer_id, customer_test in df_test.groupby(
        "Customer ID"
    ):

        customer_train = df_train[
            df_train["Customer ID"] == customer_id
        ]

        if customer_train.empty:
            continue

        # ----------------------------------------------------
        # Trigger item = most recent historical purchase
        # ----------------------------------------------------

        trigger_item = (
            customer_train
            .sort_values("Purchase Date")
            ["Item Purchased"]
            .iloc[-1]
        )

        # ----------------------------------------------------
        # Actual future items
        # ----------------------------------------------------

        actual_items = set(
            customer_test["Item Purchased"].unique()
        )

        if not actual_items:
            continue

        # ----------------------------------------------------
        # Recommendations
        # ----------------------------------------------------

        recommendations = get_recommendations(
            trigger_item,
            similarity,
            top_n=k
        )

        # ----------------------------------------------------
        # Metrics
        # ----------------------------------------------------

        precision = precision_at_k(
            recommendations,
            actual_items,
            k
        )

        recall = recall_at_k(
            recommendations,
            actual_items,
            k
        )

        actual_item = (
            customer_test
            .sort_values("Purchase Date")
            ["Item Purchased"]
            .iloc[0]
        )

        hit_rate = hit_rate_at_k(
            recommendations,
            actual_item,
            k
        )

        precisions.append(precision)
        recalls.append(recall)
        hit_rates.append(hit_rate)

        # Customer-level information only
        results.append({
            "Customer ID": customer_id,
            "Trigger": trigger_item,
            "Actual Items": list(actual_items),
            "First Actual Item": actual_item,
            "Recommendations": recommendations
        })

    # --------------------------------------------------------
    # Average metrics
    # --------------------------------------------------------

    mean_precision = np.mean(precisions) if precisions else 0.0
    mean_recall = np.mean(recalls) if recalls else 0.0
    mean_hit_rate = np.mean(hit_rates) if hit_rates else 0.0

    print(f"Average Precision@{k}: {mean_precision:.4f}")
    print(f"Average Recall@{k}:    {mean_recall:.4f}")
    print(f"Average HitRate@{k}:   {mean_hit_rate:.4f}")

    return pd.DataFrame(results)

In [31]:
results = evaluate_item_item_cf(
    df,
    k=5,
    test_size=0.20
)

Sparsity: 0.8994
Sparsity: 89.94%
Average Precision@5: 0.0994
Average Recall@5:    0.3777
Average HitRate@5:   0.3713


### Modelo de recomendación modelo de clasificación  Gradient Boosting
    Precision@:
    Recall@:
    Hitting Rate:

In [69]:
def build_customer_features(df_train):
    cat_counts = df_train.pivot_table(index='Customer ID', columns='Category', values='Item Purchased', aggfunc='count', fill_value=0)
    cat_counts.columns = [f'cat_count_{c}' for c in cat_counts.columns]
    agg = df_train.groupby('Customer ID').agg(total_purchases=('Item Purchased', 'count'), last_purchase=('Purchase Date', 'max'))
    reference_date = df_train['Purchase Date'].max()
    agg['days_since_last_purchase'] = (reference_date - agg['last_purchase']).dt.days
    agg = agg.drop(columns='last_purchase')
    return cat_counts.join(agg, how='inner')


def backtest_gradient_boosting(df, holdout_season, season_list, k=5, test_size=0.3, random_state=42):

    train_seasons = season_list[season_list != holdout_season]
    df_train = df[df['season'].isin(train_seasons)]
    df_holdout_actual = df[df['season'] == holdout_season]

    train_customers = df_train['Customer ID'].unique()
    holdout_customers = df_holdout_actual['Customer ID'].unique()
    scoring_customers = np.intersect1d(train_customers, holdout_customers)

    features = build_customer_features(df_train)
    features = features.loc[features.index.isin(scoring_customers)]

    labels = (
        df_holdout_actual[df_holdout_actual['Customer ID'].isin(features.index)]
        .groupby('Customer ID')['Category']
        .agg(lambda x: x.mode().iloc[0])
    )

    X = features
    y = labels.loc[X.index]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )

    clf = GradientBoostingClassifier(
        random_state=random_state
    )

    clf.fit(X_train, y_train)

    pred_category = pd.Series(
        clf.predict(X_test),
        index=X_test.index
    )

    item_pop_by_cat = (
        df_train.groupby(['Category', 'Item Purchased'])
        .size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
    )

    precision_scores = []
    recall_scores = []
    hit_scores = []
    recommendation_results = []

    for user in X_test.index:

        cat = pred_category[user]

        top_items = (
            item_pop_by_cat[
                item_pop_by_cat['Category'] == cat
            ]['Item Purchased']
            .head(k)
            .tolist()
        )

        actual = set(
            df_holdout_actual[
                df_holdout_actual['Customer ID'] == user
            ]['Item Purchased']
        )

        hits = len(
            set(top_items) & actual
        )

        precision = (
            hits / len(top_items)
            if len(top_items) > 0
            else 0
        )

        recall = (
            hits / len(actual)
            if len(actual) > 0
            else 0
        )

        hit = int(hits > 0)

        precision_scores.append(precision)
        recall_scores.append(recall)
        hit_scores.append(hit)

        recommendation_results.append({
            'Customer ID': user,
            'Predicted Category': cat,
            'Recommendations': top_items,
            'Actual Items': list(actual),
            'Hits': hits,
            f'Precision@{k}': precision,
            f'Recall@{k}': recall,
            f'HitRate@{k}': hit
        })

    precision_at_k = np.mean(precision_scores)
    recall_at_k = np.mean(recall_scores)
    hit_rate = np.mean(hit_scores)

    results = pd.DataFrame(
        recommendation_results
    )

    print(f'Gradient Boosting results @ {k}')
    print(f'Precision@{k}: {precision_at_k:.3f}')
    print(f'Recall@{k}:    {recall_at_k:.3f}')
    print(f'Hit Rate@{k}:  {hit_rate:.3f} ({sum(hit_scores)}/{len(hit_scores)})')

    return clf, hit_rate, precision_at_k, recall_at_k, results



In [70]:
# ============================================================
# RUN MODEL
# ============================================================

season_list = np.array([
    'winter',
    'summer',
    'monsoon',
    'post-monsoon'
])

holdout_season = 'post-monsoon'

clf, hit_rate, precision, recall, results = backtest_gradient_boosting(
    df=df,
    holdout_season=holdout_season,
    season_list=season_list,
    k=5,
    test_size=0.3,
    random_state=42
)

display(results.head(20))

Gradient Boosting results @ 5
Precision@5: 0.091
Recall@5:    0.370
Hit Rate@5:  0.422 (151/358)


,Customer ID,Predicted Category,Recommendations,Actual Items,Hits,Precision@5,Recall@5,HitRate@5
0,CUST002084,Clothing,"[Kurta, Saree, Jacket, T-shirt, Shorts]","[Running Shoes, Jeans]",0,0.0,0.000000,0
1,CUST000608,Clothing,"[Kurta, Saree, Jacket, T-shirt, Shorts]",[Kurta],1,0.2,1.000000,1
2,CUST000406,Clothing,"[Kurta, Saree, Jacket, T-shirt, Shorts]",[Socks],0,0.0,0.000000,0
3,CUST002430,Accessories,"[Backpack, Socks, Bag, Belt, Sunglasses]",[Running Shoes],0,0.0,0.000000,0
4,CUST002416,Clothing,"[Kurta, Saree, Jacket, T-shirt, Shorts]",[Kurta],1,0.2,1.000000,1
5,CUST000766,Footwear,"[Running Shoes, Sneakers, Sandals, Heels, Boots]",[Running Shoes],1,0.2,1.000000,1
6,CUST001945,Footwear,"[Running Shoes, Sneakers, Sandals, Heels, Boots]","[Backpack, Shorts]",0,0.0,0.000000,0
7,CUST001243,Clothing,"[Kurta, Saree, Jacket, T-shirt, Shorts]",[Hoodie],0,0.0,0.000000,0
8,CUST001199,Footwear,"[Running Shoes, Sneakers, Sandals, Heels, Boots]",[Sunglasses],0,0.0,0.000000,0
9,CUST003189,Clothing,"[Kurta, Saree, Jacket, T-shirt, Shorts]",[Jacket],1,0.2,1.000000,1


In [56]:
def evaluate_all_models(df, season_list, k=5):

    results = []

    # ========================================================
    # Calculate sparsity once for each holdout season
    # ========================================================

    sparsity_by_season = {}

    for holdout_season in season_list:

        train_seasons = season_list[
            season_list != holdout_season
        ]

        df_train = df[
            df['season'].isin(train_seasons)
        ]

        user_item_matrix = pd.crosstab(
            df_train['Customer ID'],
            df_train['Item Purchased']
        )

        if user_item_matrix.size > 0:
            sparsity = 1 - (
                (user_item_matrix.values > 0).sum()
                / user_item_matrix.size
            )
        else:
            sparsity = 0.0

        sparsity_by_season[holdout_season] = sparsity

    # ========================================================
    # USER-BASED
    # ========================================================

    for holdout_season in season_list:

        r = backtest_recommender_pr(
            df,
            holdout_season=holdout_season,
            season_list=season_list,
            k=k
        )

        results.append({
            'Season': holdout_season,
            'Model': 'User-based',
            'Precision': r['precision@k'].mean(),
            'Recall': r['recall@k'].mean(),
            'Hit Rate': r['hitting_rate'].mean(),
            'Sparsity': sparsity_by_season[holdout_season]
        })

    # ========================================================
    # HYBRID
    # ========================================================

    for holdout_season in season_list:

        r = backtest_hybrid_avg_threshold(
            df,
            holdout_season=holdout_season,
            season_list=season_list,
            k=k
        )

        results.append({
            'Season': holdout_season,
            'Model': 'Hybrid',
            'Precision': r['precision@k'].mean(),
            'Recall': r['recall@k'].mean(),
            'Hit Rate': r['hitting_rate'].mean(),
            'Sparsity': sparsity_by_season[holdout_season]
        })

    # ========================================================
    # POPULARITY
    # ========================================================

    for holdout_season in season_list:

        r = backtest_popularity(
            df,
            holdout_season=holdout_season,
            season_list=season_list,
            k=k
        )

        hit_rates = []

        for _, row in r.iterrows():

            recommended = row['recommended'][:k]
            actual = set(row['actual'])

            hit_rates.append(
                float(bool(set(recommended) & actual))
            )

        results.append({
            'Season': holdout_season,
            'Model': 'Popularity',
            'Precision': r['precision@k'].mean(),
            'Recall': r['recall@k'].mean(),
            'Hit Rate': np.mean(hit_rates),
            'Sparsity': sparsity_by_season[holdout_season]
        })

    # ========================================================
    # CLASSIFICATION
    # ========================================================

    for holdout_season in season_list:

        clf, hit_rate, precision, recall, r = (
            backtest_gradient_boosting(
                df,
                holdout_season=holdout_season,
                season_list=season_list,
                k=k
            )
        )

        results.append({
            'Season': holdout_season,
            'Model': 'Classification',
            'Precision': precision,
            'Recall': recall,
            'Hit Rate': hit_rate,
            'Sparsity': sparsity_by_season[holdout_season]
        })

    # ========================================================
    # RESULTS PER SEASON
    # ========================================================

    results_df = pd.DataFrame(results)

    # ========================================================
    # AVERAGE ACROSS ALL SEASONS
    # ========================================================

    evaluation_matrix = (
        results_df
        .groupby('Model')[
            ['Precision', 'Recall', 'Hit Rate', 'Sparsity']
        ]
        .mean()
        .reset_index()
    )

    # ========================================================
    # PRINT FINAL EVALUATION MATRIX
    # ========================================================

    print("\n=== MODEL EVALUATION — AVERAGE ACROSS SEASONS ===")
    display(
        evaluation_matrix.round(4)
    )

    return results_df, evaluation_matrix

In [71]:
results_by_season, evaluation_matrix = evaluate_all_models(
    df,
    season_list,
    k=5
)

Sparsity: 0.942
Precision@5: 0.072
Recall@5:    0.226
hitting rate :0.285
Sparsity: 0.943
Precision@5: 0.075
Recall@5:    0.227
hitting rate :0.295
Sparsity: 0.944
Precision@5: 0.076
Recall@5:    0.233
hitting rate :0.315
Sparsity: 0.941
Precision@5: 0.055
Recall@5:    0.176
hitting rate :0.210
threshold = 1.0 (avg of top-5 neighbors, exact matches only)
Precision@5: 0.0909
Recall@5:    0.3041
hitting rate : 0.3695
customers routed to CF:                830
customers routed to popularity fallback: 702

sparsity by season:
  summer: 0.9422
  monsoon: 0.9367
  post-monsoon: 0.9484
overall sparsity (avg across seasons): 0.9424
threshold = 1.0 (avg of top-5 neighbors, exact matches only)
Precision@5: 0.0899
Recall@5:    0.2983
hitting rate : 0.3746
customers routed to CF:                924
customers routed to popularity fallback: 675

sparsity by season:
  winter: 0.9428
  monsoon: 0.9367
  post-monsoon: 0.9484
overall sparsity (avg across seasons): 0.9426
threshold = 1.0 (avg of top-5 ne

,Model,Precision,Recall,Hit Rate,Sparsity
0,Classification,0.0958,0.3515,0.4342,0.9039
1,Hybrid,0.0897,0.2983,0.3691,0.9039
2,Popularity,0.1102,0.4028,0.4873,0.9039
3,User-based,0.0693,0.2158,0.2759,0.9039


In [ ]:
from reactiva.recommender1 import get_recommendations_items

ModuleNotFoundError: No module named 'reactiva.recommender1'

In [90]:
reactiva.recommender1

AttributeError: module 'reactiva' has no attribute 'recommender1'